# HJB Matching — Toy Experiments

Run all toy experiments (4 Gaussians, Two Moons, Swiss Roll, Lensing) on Colab GPU.

Outputs (weights + loss history) are saved to `toy/outputs/<experiment>/` and can be downloaded for local figure generation.

## Setup

In [ ]:
# Clone the repo
!git clone https://github.com/sumit-sinha-seas/HJB_matching.git
%cd HJB_matching

In [ ]:
# Install dependencies
!pip install -q jax jaxlib dm-haiku optax pyyaml matplotlib

In [ ]:
import os
import pickle
import yaml
import jax
import jax.numpy as jnp
import optax

from toy.distributions import DISTRIBUTIONS, nu_lensing
from toy.model import build_w_net
from toy.train import build_train_fns

print(f"JAX devices: {jax.devices()}")

## Training Loop

In [ ]:
def run_experiment(config_path, output_dir):
    """Train a toy experiment and save outputs."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    os.makedirs(output_dir, exist_ok=True)

    N = cfg['N']
    key = jax.random.PRNGKey(0)

    init_sample = DISTRIBUTIONS[cfg['distribution']]
    nu_fn = nu_lensing if cfg.get('use_analytical_nu', False) else None
    target_pos = jnp.array([cfg['target_pos']])

    w_net = build_w_net(cfg)
    key, subkey = jax.random.split(key)
    w_params = w_net.init(subkey, jnp.zeros((N, 2)), jnp.full((N,), 0, dtype=jnp.int32))

    w_opt = optax.adam(cfg['lr'])
    w_opt_state = w_opt.init(w_params)

    fns = build_train_fns(w_net, w_opt, cfg, nu_fn=nu_fn)
    train_step = fns['train_step']

    loss_lst = []
    log_every = cfg.get('log_every', 50)

    for epoch in range(cfg['total_epochs']):
        key, subkey1, subkey2 = jax.random.split(key, 3)
        pos0 = init_sample(subkey1, N)

        w_params, w_opt_state, loss, loss1, loss2, loss3, final_pos, _ = train_step(
            w_params, w_opt_state, pos0, subkey2, target_pos
        )
        loss_lst.append(float(loss))

        if epoch % log_every == 0:
            print(f"  Epoch {epoch:4d}  loss={loss:.6f}  l1={loss1:.4f}  l2={loss2:.4f}  l3={loss3:.4f}")

    # Save outputs
    with open(f'{output_dir}/w_params.pkl', 'wb') as f:
        pickle.dump(w_params, f)
    with open(f'{output_dir}/loss_lst.pkl', 'wb') as f:
        pickle.dump(loss_lst, f)
    with open(f'{output_dir}/config_used.yaml', 'w') as f:
        yaml.dump(cfg, f)

    print(f"  Done. Final loss={loss_lst[-1]:.6f}. Saved to {output_dir}/")
    return w_params, loss_lst, cfg

## Run Experiments

Each cell runs one experiment. Comment out any you don't need.

In [ ]:
print("=" * 50)
print("  4 Gaussians")
print("=" * 50)
run_experiment('toy/configs/4gaussian.yaml', 'toy/outputs/4gaussians')

In [ ]:
print("=" * 50)
print("  Two Moons")
print("=" * 50)
run_experiment('toy/configs/two_moon.yaml', 'toy/outputs/two_moon')

In [ ]:
print("=" * 50)
print("  Swiss Roll")
print("=" * 50)
run_experiment('toy/configs/swissroll.yaml', 'toy/outputs/swiss_roll')

In [ ]:
print("=" * 50)
print("  Lensing")
print("=" * 50)
run_experiment('toy/configs/lensing.yaml', 'toy/outputs/lensing')

## Download Outputs

Zip the outputs for local download (for figure generation with `toy/fig_paper.py`).

In [ ]:
!zip -r toy_outputs.zip toy/outputs/

# On Colab, trigger download:
try:
    from google.colab import files
    files.download('toy_outputs.zip')
except ImportError:
    print("Not running on Colab. Find toy_outputs.zip in the working directory.")